In [ ]:
import os
import re
from time import sleep
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from bs4 import BeautifulSoup


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser

from pydantic import BaseModel, Field

MODEL = "gemma-4-31b-it"
TEMPERATURE = 0

In [ ]:
class TarjetaGrafica(BaseModel):
    nombre: str = Field(description="nombre de la tarjeta grafica")
    marca: str = Field(description="marca de la tarjeta grafica")
    precio_actual: float = Field(description="precio de la tarjeta grafica con descuento actual")
    precio_fijo: float = Field(description="precio de la tarjeta grafica din descuento")
    stock: str = Field(description="confirmacion de si hay disponibilidad del producto")
    valoracion: float = Field(description="puntaje segun la valoracion de los clientes")
    total_opiniones: int = Field(description="cantidad de valoraciones")
    opiniones_usuario: list[str] = Field(description="lista de opiniones de usuario")
    porcentaje_recomendacion: float = Field(description="porcentaje de recomendacion")
    media_estrellas: float = Field(description="media de estrellas en base a opiniones")
    url: str = Field(description="URL completa de la página web del producto")
    info_descriptiva: dict[str, str] = Field(description="diccionario que contenga todas las especificaciones tecnicas de la tarjeta grafica")

parser = PydanticOutputParser(pydantic_object=TarjetaGrafica)

In [ ]:
def limpiar_html(ruta_archivo):
    with open(ruta_archivo,"r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
        texto_limpio = soup.get_text(strip=True)
    return texto_limpio

In [ ]:
def preparar_contenido_html():
    rutas_tarjetas = []
    carpeta = "data_raw_tarjetas"
    todos_los_archivos = os.listdir(carpeta)

    for archivo in todos_los_archivos:
        if archivo.endswith(".html"):
            ruta_completa = os.path.join(carpeta, archivo)
            rutas_tarjetas.append(ruta_completa)
            
    htmls = []
    for ruta in rutas_tarjetas:
        htmls.append(limpiar_html(ruta))
        
    return htmls

In [ ]:
def ejecutar_extraccion_ia(htmls):
    load_dotenv()
    
    llm = ChatGoogleGenerativeAI(model=MODEL, temperature=TEMPERATURE)

    template = """
    Eres un expert extraccion de datos. analiza la siguiente pagina web de una tarjeta . Si algún dato no está explícito en la web, infiérelo si el contexto es claro, o usa los valores por defecto indicados en las instrucciones.
    Dame la respuesta en español.
    {instrucciones_formato}

    Tarjeta grafica:
    {tarjeta_grafica}
    """
    prompt = ChatPromptTemplate.from_template(template=template)
    cadena = prompt | llm | parser

    respuestas = []
    
    for html in htmls[0:500]:
        llm_input = {
            "tarjeta_grafica": html,
            "instrucciones_formato": parser.get_format_instructions()
        }

        respuesta = cadena.invoke(llm_input)
        respuestas.append(respuesta.model_dump())
        print(len(respuestas))
        
    return respuestas

In [ ]:
def transformar_a_dataframe(respuestas):
   
    df_tgraf = pd.json_normalize(respuestas)

    df_tgraf = df_tgraf.rename(columns={"nombre": "modelo"})
    df_tgraf = df_tgraf.rename(columns={"precio_actual": "precio"})
    df_tgraf = df_tgraf.rename(columns={"total_opiniones": "numero_total_opiniones"})
    df_tgraf = df_tgraf.rename(columns={"opiniones_usuario": "texto_resena"})
    df_tgraf = df_tgraf.rename(columns={"media_estrellas": "valoracion_media"})

    df_tgraf[['reloj_base', 'reloj_boost']] = df_tgraf['info_descriptiva.Frecuencia base / boost'].str.extract(r'(.+?)\s*/\s*(.+)')
    df_tgraf[['memoria_vram', 'tipo_memoria']] = df_tgraf['info_descriptiva.Memoria VRAM'].str.split(' ', n=1, expand=True)

    df_tgraf = df_tgraf.rename(columns={"info_descriptiva.Ancho de bus": "bus_memoria"})
    df_tgraf = df_tgraf.rename(columns={"info_descriptiva.Velocidad de transferencia": "velocidad_memoria"})
    df_tgraf = df_tgraf.rename(columns={"info_descriptiva.Resolución máxima": "resolucion_maxima"})
    df_tgraf = df_tgraf.rename(columns={"info_descriptiva.Salidas de vídeo": "salidas_video"})
    df_tgraf = df_tgraf.rename(columns={'ancho_banda_memoria': "ancho_banda"})

    df_tgraf['categoria'] = 'tarjeta_grafica'
    df_tgraf['numero_recomendaciones'] = df_tgraf['numero_total_opiniones'] * (df_tgraf['porcentaje_recomendacion'] / 100)
    df_tgraf['fuente_desglose'] = 'media_threshold_classification'
    df_tgraf['id_producto'] = range(1, len(df_tgraf) + 1)

    df_tgraf['estrellas_1'] = (df_tgraf['valoracion_media'] < 1).astype(int)
    df_tgraf['estrellas_2'] = (df_tgraf['valoracion_media'] < 2).astype(int)
    df_tgraf['estrellas_3'] = (df_tgraf['valoracion_media'] < 3).astype(int)
    df_tgraf['estrellas_4'] = (df_tgraf['valoracion_media'] < 4).astype(int)
    df_tgraf['estrellas_5'] = (df_tgraf['valoracion_media'] < 5).astype(int)

In [ ]:
htmls_limpios = preparar_contenido_html()


respuestas_crudas = ejecutar_extraccion_ia(htmls_limpios)

df_tarjetas_final = transformar_a_dataframe(respuestas_crudas)